## Daemon troubleshooting

When "Docker is broken," a small, fixed set of commands answers most questions — starting with the daemon itself, the process every other command depends on.

**Is it healthy, and what's it doing?**

```bash
docker version          # client + server — if server errors, the daemon is down
docker info             # every current setting: drivers, root dir, counts
docker system df        # disk-usage breakdown (add -v for per-object detail)
docker system events    # live stream of everything the daemon does
```

`docker version` is the fastest triage: a client with **no server** means the daemon isn't running or you can't reach its socket. `docker system df` plus `prune` is how you reclaim the disk that Docker quietly fills.

**When the daemon won't start**, drop to systemd and its logs:

```bash
sudo systemctl status docker         # is it running? why did it stop?
sudo journalctl -u docker -n 200     # last 200 lines of daemon log
sudo dockerd --debug                 # run in the foreground with debug output
```

`journalctl -u docker` is where the *real* error lives — a Go stack trace or a plain "unknown config key." Running `dockerd --debug` in the foreground surfaces a startup failure immediately instead of systemd swallowing it.

**The two causes that account for most "won't start" cases:**

- **A bad `daemon.json`** — a syntax error or unknown key. `sudo dockerd --validate` checks it *without* starting the daemon (the habit from the config section).
- **A storage-driver mismatch** — `/var/lib/docker` was created by one driver and the config now names another; the daemon refuses to start. Either restore the original driver or move the data dir aside.

The method generalises: **is the process up (`systemctl status`) → what did it say (`journalctl`) → what's misconfigured (`--validate`, `info`).** That same up → logs → config loop is the backbone of the container and network playbooks next.